# Python Exception Handling

## 1. Errors vs Exceptions

Syntax errors are detected while parsing code. Exceptions occur during execution. An unhandled exception produces a traceback and propagates outward.

In [1]:
try:
    10 / 0
except ZeroDivisionError as exc:
    print(type(exc).__name__)
    print(exc)

ZeroDivisionError
division by zero


## 2. Basic `try` / `except`

Put potentially failing code in `try`. A matching `except` handler runs when an exception occurs.

In [2]:
try:
    value = int("abc")
except ValueError:
    print("Conversion failed.")

Conversion failed.


## 3. Capturing the Exception

`except ExceptionType as exc` gives access to the exception object, including its message and `args`.

In [3]:
try:
    int("abc")
except ValueError as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(exc.args)

ValueError
invalid literal for int() with base 10: 'abc'
("invalid literal for int() with base 10: 'abc'",)


## 4. Multiple `except` Clauses

Handlers are checked in order. The first compatible handler is used. Put specific exceptions before broader exception classes.

In [4]:
def divide(value):
    try:
        return 10 / value
    except ZeroDivisionError:
        return "cannot divide by zero"
    except TypeError:
        return "value is not numeric"

print(divide(0))
print(divide("x"))

cannot divide by zero
value is not numeric


## 5. Multiple Exception Types in One Handler

An `except` clause can match a tuple of exception classes.

In [5]:
try:
    int("abc")
except (ValueError, TypeError):
    print("Conversion failed.")

Conversion failed.


## 6. Bare `except`

A bare `except` catches any exception. Use it cautiously because it also catches exceptions outside the normal `Exception` branch, such as `KeyboardInterrupt` and `SystemExit`.

In [6]:
try:
    raise ValueError("example")
except:
    print("Caught it.")

Caught it.


## 7. `except Exception`

`Exception` is the usual base class for non-system-exiting application exceptions and for user-defined exceptions.

In [7]:
try:
    {}["missing"]
except Exception as exc:
    print(type(exc).__name__, exc)

KeyError 'missing'


## 8. `else`

The `else` block runs only when the `try` suite completes without an exception. Exceptions raised inside `else` are not handled by the preceding `except` clauses.

In [8]:
try:
    value = int("42")
except ValueError:
    print("Invalid integer")
else:
    print("Success:", value)

Success: 42


## 9. `finally`

`finally` is used for cleanup and runs whether or not an exception occurs.

In [9]:
try:
    print("try")
    1 / 0
except ZeroDivisionError:
    print("except")
finally:
    print("finally")

try
except
finally


## 10. `try` + `finally`

A `try` statement may contain only `finally` when the exception should propagate but cleanup must still happen.

In [10]:
try:
    print("work")
finally:
    print("cleanup")

work
cleanup


## 11. Full `try` / `except` / `else` / `finally`

Typical flow:

```text
try
 ├─ exception → matching except → finally
 └─ no exception → else → finally
```

In [11]:
try:
    value = int("10")
except ValueError:
    print("except")
else:
    print("else:", value)
finally:
    print("finally")

else: 10
finally


## 12. `raise`

`raise` explicitly raises an exception. It can raise an exception class or an exception instance.

In [12]:
def withdraw(balance, amount):
    if amount > balance:
        raise ValueError("Insufficient balance")
    return balance - amount

try:
    print(withdraw(100, 150))
except ValueError as exc:
    print(type(exc).__name__, exc)

ValueError Insufficient balance


## 13. Re-raising

A bare `raise` inside an active exception handler re-raises the currently handled exception.

In [13]:
try:
    1 / 0
except ZeroDivisionError:
    print("Handling locally...")
    try:
        raise
    except ZeroDivisionError as exc:
        print("Re-raised:", type(exc).__name__)

Handling locally...
Re-raised: ZeroDivisionError


## 14. Exception Chaining

`raise NewException(...) from original_exception` explicitly sets the original exception as the new exception's cause. `from None` suppresses display of the implicit context.

In [14]:
def convert(value):
    try:
        return int(value)
    except ValueError as exc:
        raise RuntimeError("Conversion failed") from exc

try:
    convert("abc")
except RuntimeError as exc:
    print(type(exc).__name__)
    print("cause:", type(exc.__cause__).__name__)

RuntimeError
cause: ValueError


## 15. Exception Context

When a new exception is raised while another is being handled, Python records the earlier exception in `__context__` unless explicit chaining changes the relationship.

In [15]:
try:
    try:
        1 / 0
    except ZeroDivisionError:
        raise ValueError("new error")
except ValueError as exc:
    print("current:", type(exc).__name__)
    print("context:", type(exc.__context__).__name__)

current: ValueError
context: ZeroDivisionError


## 16. Exception Attributes

Useful attributes include `args`, `__traceback__`, `__context__`, `__cause__`, and `__suppress_context__`.

Python 3.11+ also provides `add_note()` and `__notes__`.

In [16]:
try:
    raise ValueError("bad input")
except ValueError as exc:
    print("args:", exc.args)
    print("traceback exists:", exc.__traceback__ is not None)
    exc.add_note("Check the input source.")
    print("notes:", exc.__notes__)

args: ('bad input',)
traceback exists: True
notes: ['Check the input source.']


## 17. User-Defined Exceptions

Create application-specific exceptions by subclassing `Exception` or one of its subclasses. 

In [17]:
class InvalidAgeError(Exception):
    pass

def register(age):
    if age < 0:
        raise InvalidAgeError("Age cannot be negative")
    return age

try:
    register(-1)
except InvalidAgeError as exc:
    print(type(exc).__name__, exc)

InvalidAgeError Age cannot be negative


## 18. Custom Exception Data

A custom exception can store structured information. Call `super().__init__()` so the base exception receives the message.

In [20]:
class ValidationError(Exception):
    def __init__(self, field, message):
        self.field = field
        self.message = message
        super().__init__(message)

try:
    raise ValidationError("email", "Invalid email")
except ValidationError as exc:
    print(exc.field)
    print(exc.message)
    print(exc.args)

email
Invalid email
('Invalid email',)


## 19. Important Built-in Exceptions

| Exception | Typical cause |
|---|---|
| `TypeError` | Unsupported operation / wrong type |
| `ValueError` | Right type, inappropriate value |
| `IndexError` | Sequence index out of range |
| `KeyError` | Mapping key not found |
| `AttributeError` | Attribute lookup fails |
| `NameError` | Name not found |
| `UnboundLocalError` | Local variable referenced before assignment |
| `ZeroDivisionError` | Division/modulo by zero |
| `FileNotFoundError` | File or directory does not exist |
| `PermissionError` | Insufficient access rights |
| `ImportError` | Import cannot be completed |
| `ModuleNotFoundError` | Module/package cannot be found |
| `AssertionError` | `assert` condition is false |
| `RuntimeError` | Generic runtime problem |



## 20. `assert`

`assert condition` raises `AssertionError` when the condition is false. Assertions are intended for internal assumptions/debugging, not for validating untrusted input.

In [24]:
x = 10

assert x > 0

try:
    assert x < 0, "x must be negative"
except AssertionError as exc:
    print(type(exc).__name__, exc)

AssertionError x must be negative
